# LLM Engineering Decisions — Token Economics & Context Management for Northstar

## Why this companion notebook exists

The main S03 lab explains tokens, context windows, and streaming. This companion turns those mechanisms into measured engineering decisions for Northstar's 50,000-ticket-per-day workload: how much a design costs, what happens when context does not fit, and when streaming is worth its complexity.

Each lab ends with a decision rather than only an observation. You will compare word and token counts, build a cost model, apply a context-selection policy, and contrast interactive and batch delivery.

## What you will learn

1. Demonstrate why word count is not a reliable token budget.
2. Calculate per-attempt, per-ticket, daily, and monthly model cost.
3. Measure the token contribution of each request component.
4. Apply a selection policy when all available context does not fit.
5. Decide when streaming improves the user experience and when it only adds complexity.
6. Write a business-facing recommendation supported by explicit assumptions.

## How the labs fit together

Labs 1–3 use tokenization and ordinary Python to study budgeting and selection. Lab 4 loads the small teaching model only when generation is needed. The context-selection example is deliberately a simple keyword heuristic, not a retrieval pipeline; its purpose is to make the budgeting decision visible and explainable.


## Load the tokenizer

The tokenizer is enough for Labs 1–3 because those labs measure text and apply budgeting rules without generating a response. The model itself is loaded later in Lab 4.


In [ ]:
import json
import re
import sys
import time
from transformers import AutoTokenizer

# Defensive: Lab 1 prints non-Latin-script and emoji samples; some terminals/
# notebook kernels report a non-UTF-8 stdout encoding, which would otherwise
# raise UnicodeEncodeError on those prints.
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except (AttributeError, ValueError):
    pass

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"  # same model S03 uses -- no reason to change it
print("Loading tokenizer only:", MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Tokenizer ready. The full model has NOT been loaded yet -- Labs 1-3 don't need it.")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


---
# Lab 1 — Token Exploration

**Goal:** stop estimating cost and budget from word count. Build your own intuition for
*when* token count and word count diverge, using direct comparisons.


In [ ]:
samples = {
    "Plain English":            "The ticket was resolved after one follow-up call.",
    "Heavy punctuation":        "Wait... this is URGENT!! Why?! Fix it NOW!!!",
    "Uncommon / technical word": "The deserialization raised an UnrecognizedTokenException.",
    "Short code snippet":       "if ticket.priority == 'high' and not ticket.assigned_agent: escalate(ticket)",
    "Multilingual (French)":    "Le ticket a ete resolu apres un seul appel de suivi.",
    "Multilingual (Hindi)":     "\u091f\u093f\u0915\u091f \u0928\u093f\u092a\u091f \u0915\u0947 \u092c\u093e\u0926 \u0939\u0932 \u0939\u0941\u0906\u0964",
    "Repetitive common words":  "the ticket the ticket the ticket the ticket the ticket",
}

rows = []
for label, text in samples.items():
    n_tokens = count_tokens(text)
    n_words = len(text.split())
    rows.append((label, n_words, n_tokens, round(n_tokens / max(n_words, 1), 2)))

print(f"{'Sample':28s} {'words':>6s} {'tokens':>7s} {'tokens/word':>12s}")
for label, n_words, n_tokens, ratio in rows:
    print(f"{label:28s} {n_words:6d} {n_tokens:7d} {ratio:12.2f}")


**Expected pattern (not exact numbers — this depends on the tokenizer you're running):**
the plain-English and repetitive-common-words rows should land close to **1 token per
word**; the heavy-punctuation, technical-word, code, and non-Latin-script rows should all
show a **noticeably higher tokens-per-word ratio** — some of that punctuation, the
technical term, the code symbols, and especially the Hindi script typically split into
several sub-word tokens each, even though none of those rows has unusually many *words*.

> [!IMPORTANT]
> **⚙️ Engineering implication:** "this ticket is short" (few words) does **not** mean
> "this ticket is cheap" (few tokens). A short, code-heavy, or non-English ticket can cost
> more tokens than a longer plain-English one. Any token budget estimated from word count
> alone is a guess, not a measurement.

> [!NOTE]
> **(optional) Compare tokenizers directly.** Different models' tokenizers split the same
> text differently. Skip this cell on the required path; it reinforces the point above
> with a second tokenizer, not a new one.


In [ ]:
# OPTIONAL: compare the examples with a second tokenizer.
import tiktoken
other_tokenizer = tiktoken.get_encoding("cl100k_base")

for label, text in samples.items():
    local_count = count_tokens(text)
    other_count = len(other_tokenizer.encode(text))
    print(f"{label:28s} SmolLM2={local_count:3d}  cl100k_base={other_count:3d}")
print("\nCounts differ by tokenizer -- a token budget computed for one model")
print("does not automatically transfer to a different model's tokenizer.")


### ✍️ Learner exercise

Write two strings with the **same word count** where one costs noticeably more tokens
than the other. State which property of tokenization you're exploiting (punctuation
density, uncommon vocabulary, code syntax, or script).

<details>
<summary>Instructor solution</summary>

Example pair (5 words each): `"The cat sat on it."` vs. `"Deserialize asynchronous
polymorphic microservice architectures."` — same word count, but the second string uses
uncommon, multi-syllable technical vocabulary that a general-purpose tokenizer is far
less likely to have single-token entries for, so it should tokenize into noticeably more
pieces per word than the first. Any pair that holds word count constant while varying
vocabulary commonality, punctuation density, or script is an acceptable answer — the
specific words don't matter, the demonstrated property does.

</details>


---
# Lab 2 — Token Economics

**Northstar processes 50,000 support tickets per day.** This lab turns that one fact,
plus Lab 1's token-counting habit, into an actual cost model you can vary and interrogate.

> [!WARNING]
> Every price in this lab is an **illustrative assumption**, clearly labeled as such —
> not a live quote from any provider. Treat every dollar figure below as "what it would
> cost *if* these prices and these assumptions held," not a measured bill.


In [ ]:
def ticket_cost_model(
    avg_input_tokens: float,
    avg_output_tokens: float,
    price_per_million_input: float,
    price_per_million_output: float,
    daily_ticket_volume: int,
    retry_rate: float,
    cached_input_tokens: float = 0.0,
    cached_price_per_million_input: float | None = None,
    days_per_month: float = 30.44,
) -> dict:
    '''A deterministic cost model -- every output is a direct function of the inputs
    you supply, none of it is a measured bill. See the module docstring-style warning
    above: prices here are illustrative assumptions, not live market data.

    retry_rate: the probability a single attempt is NOT sufficient and must be retried
    (0.0 <= retry_rate < 1.0). Expected attempts per eventual success follows a geometric
    expectation: 1 / (1 - retry_rate).
    '''
    if not (0 <= retry_rate < 1):
        raise ValueError("retry_rate must be in [0, 1) -- 1.0 would mean infinite retries")
    if cached_input_tokens > avg_input_tokens:
        raise ValueError("cached_input_tokens cannot exceed avg_input_tokens")

    if cached_price_per_million_input is None:
        cached_price_per_million_input = price_per_million_input

    fresh_input_tokens = avg_input_tokens - cached_input_tokens
    cost_per_attempt = (
        fresh_input_tokens * price_per_million_input
        + cached_input_tokens * cached_price_per_million_input
        + avg_output_tokens * price_per_million_output
    ) / 1_000_000

    expected_attempts_per_success = 1.0 / (1.0 - retry_rate)
    cost_per_completed_ticket = cost_per_attempt * expected_attempts_per_success
    daily_cost = cost_per_completed_ticket * daily_ticket_volume
    monthly_cost = daily_cost * days_per_month

    return {
        "cost_per_attempt_usd": round(cost_per_attempt, 6),
        "expected_attempts_per_success": round(expected_attempts_per_success, 4),
        "cost_per_completed_ticket_usd": round(cost_per_completed_ticket, 6),
        "daily_cost_usd": round(daily_cost, 2),
        "monthly_cost_usd": round(monthly_cost, 2),
    }

NORTHSTAR_DAILY_TICKETS = 50_000


## Two worked scenarios, same volume, different context habits

**Scenario A — Lean:** agents attach only the current ticket text and a short system
instruction (~250 input tokens), a brief classification output (~40 output tokens), and a
low retry rate (2%) because the short, focused input rarely trips validation.

**Scenario B — Context-heavy:** agents attach the full prior conversation thread plus a
support-doc excerpt (~1,200 input tokens), a slightly longer output (~80 tokens), and a
higher retry rate (8%) because the longer, noisier input produces more validation
failures.


In [ ]:
scenario_a_lean = ticket_cost_model(
    avg_input_tokens=250, avg_output_tokens=40,
    price_per_million_input=1.00, price_per_million_output=4.00,
    daily_ticket_volume=NORTHSTAR_DAILY_TICKETS, retry_rate=0.02,
)
scenario_b_heavy = ticket_cost_model(
    avg_input_tokens=1200, avg_output_tokens=80,
    price_per_million_input=1.00, price_per_million_output=4.00,
    daily_ticket_volume=NORTHSTAR_DAILY_TICKETS, retry_rate=0.08,
)

print("Scenario A -- Lean:")
for k, v in scenario_a_lean.items():
    print(f"  {k}: {v}")
print("\nScenario B -- Context-heavy:")
for k, v in scenario_b_heavy.items():
    print(f"  {k}: {v}")

delta = scenario_b_heavy["monthly_cost_usd"] - scenario_a_lean["monthly_cost_usd"]
print(f"\nMonthly cost difference: ${delta:,.2f} -- same ticket volume, same model, "
      f"only the attached context and resulting retry rate differ.")


**Observation.** The same 50,000-ticket/day volume, same model, same per-token prices —
the *only* difference between the two scenarios is how much context gets attached per
ticket and the retry rate that habit produces. That alone should produce a real monthly
gap, not a rounding difference.

> [!IMPORTANT]
> **⚙️ Engineering implication:** "context habits" are a cost lever, independent of model
> choice or pricing. Before recommending a cheaper *model*, check whether the same savings
> are available for free by trimming what gets attached per call — Lab 3 builds exactly
> that trimming policy.

### Sensitivity: reducing prompt size, retries, and output length

Each cell below holds every parameter from Scenario B fixed except one, so you can read
the cost effect of that one change directly.


In [ ]:
# Effect of reducing prompt size alone (retry rate and output length held at Scenario B's values)
for input_tokens in [1200, 900, 600, 300]:
    result = ticket_cost_model(
        avg_input_tokens=input_tokens, avg_output_tokens=80,
        price_per_million_input=1.00, price_per_million_output=4.00,
        daily_ticket_volume=NORTHSTAR_DAILY_TICKETS, retry_rate=0.08,
    )
    print(f"avg_input_tokens={input_tokens:5d} -> monthly cost ${result['monthly_cost_usd']:>10,.2f}")


In [ ]:
# Effect of retry rate alone (input/output tokens held at Scenario B's values)
for retry_rate in [0.02, 0.08, 0.15, 0.30]:
    result = ticket_cost_model(
        avg_input_tokens=1200, avg_output_tokens=80,
        price_per_million_input=1.00, price_per_million_output=4.00,
        daily_ticket_volume=NORTHSTAR_DAILY_TICKETS, retry_rate=retry_rate,
    )
    print(f"retry_rate={retry_rate:.2f} -> expected attempts/success "
          f"{result['expected_attempts_per_success']:.3f} -> monthly cost ${result['monthly_cost_usd']:>10,.2f}")


In [ ]:
# Effect of output-token limit alone (input tokens/retry rate held at Scenario B's values)
for output_tokens in [40, 80, 150, 300]:
    result = ticket_cost_model(
        avg_input_tokens=1200, avg_output_tokens=output_tokens,
        price_per_million_input=1.00, price_per_million_output=4.00,
        daily_ticket_volume=NORTHSTAR_DAILY_TICKETS, retry_rate=0.08,
    )
    print(f"avg_output_tokens={output_tokens:4d} -> monthly cost ${result['monthly_cost_usd']:>10,.2f}")


**Observation.** All three levers — prompt size, retry rate, and output-token limit —
move monthly cost, but notice retry rate moves it **multiplicatively** (every retry
re-pays the full input cost, not just the output), while output-token limit moves it only
through the (usually cheaper, but not free) output price. A high retry rate is often the
most expensive lever to ignore, not the most obvious one.

> [!NOTE]
> **(optional) Cached-input pricing.** Some providers discount a repeated, eligible
> prompt prefix (e.g., a stable system instruction). The cell below compares identical
> scenarios with and without a cached-prefix discount applied to part of the input.


In [ ]:
# OPTIONAL
no_cache = ticket_cost_model(
    avg_input_tokens=600, avg_output_tokens=60,
    price_per_million_input=1.00, price_per_million_output=4.00,
    daily_ticket_volume=NORTHSTAR_DAILY_TICKETS, retry_rate=0.03,
)
with_cache = ticket_cost_model(
    avg_input_tokens=600, avg_output_tokens=60,
    price_per_million_input=1.00, price_per_million_output=4.00,
    daily_ticket_volume=NORTHSTAR_DAILY_TICKETS, retry_rate=0.03,
    cached_input_tokens=400, cached_price_per_million_input=0.25,
)
print("Without prefix caching, monthly cost: $", no_cache["monthly_cost_usd"])
print("With 400 of 600 input tokens cached, monthly cost: $", with_cache["monthly_cost_usd"])
print("\nAny caching saving depends entirely on the provider's actual eligibility rules")
print("and reported usage -- never assume a discount applies without verifying it live.")


## 🛠️ Interactive exercise

Vary the parameters yourself and observe which assumptions move monthly cost most. If interactive widgets are available, the cell displays sliders; otherwise it prints one direct calculation so the analysis remains usable.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def _report(avg_input_tokens, avg_output_tokens, price_per_million_input,
                price_per_million_output, daily_ticket_volume, retry_rate):
        result = ticket_cost_model(
            avg_input_tokens=avg_input_tokens, avg_output_tokens=avg_output_tokens,
            price_per_million_input=price_per_million_input,
            price_per_million_output=price_per_million_output,
            daily_ticket_volume=daily_ticket_volume, retry_rate=retry_rate,
        )
        for k, v in result.items():
            print(f"{k}: {v}")

    widgets.interact(
        _report,
        avg_input_tokens=widgets.IntSlider(value=600, min=50, max=4000, step=50),
        avg_output_tokens=widgets.IntSlider(value=80, min=10, max=1000, step=10),
        price_per_million_input=widgets.FloatSlider(value=1.00, min=0.05, max=10.0, step=0.05),
        price_per_million_output=widgets.FloatSlider(value=4.00, min=0.05, max=20.0, step=0.05),
        daily_ticket_volume=widgets.IntSlider(value=50_000, min=1_000, max=200_000, step=1_000),
        retry_rate=widgets.FloatSlider(value=0.05, min=0.0, max=0.9, step=0.01),
    )
except ImportError:
    print("ipywidgets not available in this environment -- falling back to one direct call:")
    print(ticket_cost_model(
        avg_input_tokens=600, avg_output_tokens=80,
        price_per_million_input=1.00, price_per_million_output=4.00,
        daily_ticket_volume=NORTHSTAR_DAILY_TICKETS, retry_rate=0.05,
    ))


### ✍️ Learner exercise

Using the sliders (or direct calls), find the single parameter change that most reduces
monthly cost **without changing daily ticket volume**. Which lever did you pick, and why
did it move the number the most?

<details>
<summary>Instructor solution</summary>

Usually retry rate, for the reason named in the Observation above: retries multiply the
*entire* per-attempt cost (input and output tokens both), while reducing input tokens
only reduces the input portion, and reducing output tokens only reduces the (often
cheaper) output portion. A learner who identifies retry rate as the highest-leverage,
lowest-obvious lever has understood the model, not just run it.

</details>


---
# Lab 3 — Context Window Engineering

**Goal:** build a realistic Northstar request out of its real parts, measure what each
part actually costs in tokens, and watch — not just read about — what happens when they
don't all fit.


In [ ]:
request_parts = {
    "system": "You are a Northstar support assistant. Answer using only the provided context.",
    "ticket": "What is this customer's guaranteed SLA response time?",
    "output_instructions": "Respond with the SLA time and the tier it is based on.",
    "history_turn_1": "Customer: We are on the Enterprise plan.",
    "doc_sla_table": "SLA table: Enterprise 1 hour, Pro 4 hours, Standard 24 hours.",
    "doc_billing_faq": "Billing FAQ: invoices are issued monthly on the first business day.",
}

print(f"{'Component':22s} {'tokens (live count)':>20s}")
live_counts = {}
for key, text in request_parts.items():
    n = count_tokens(text)
    live_counts[key] = n
    print(f"{key:22s} {n:20d}")
print(f"\n{'TOTAL if everything is included':22s} {sum(live_counts.values()):20d}")


**Observation.** Six small, individually reasonable pieces already add up to a real
number before a single output token exists. Every one of these pieces is competing for
the *same* context-window budget S03 §6 introduced.

## Budget allocation: what happens when it doesn't all fit

Suppose this request has to fit inside a **130-token input budget** (deliberately small,
so the mechanism is unmistakable — a production budget would be larger, the mechanism is
identical). `system`, `ticket`, and `output_instructions` are **mandatory** — they define
what the model is even being asked to do, so they're never candidates for pruning.
Everything else competes for whatever budget remains.

> [!NOTE]
> The token counts below are **fixed, illustrative values**, not the live counts from the
> cell above — chosen deliberately so this specific demonstration reliably shows the
> mechanism regardless of which tokenizer you're running, the same deterministic-by-design
> choice **S01**'s mock model makes for its own teaching demo. The live-counted numbers
> above are the real ones for *this* tokenizer; the fixed numbers below are what makes
> *this specific contrast* land every time.


In [ ]:
def select_within_budget(blocks, available_budget, mandatory_keys, priority_order):
    '''blocks: dict[key] -> (token_count:int, text:str).
    mandatory_keys: always included; raises if they alone exceed budget.
    priority_order: remaining keys, highest priority first (kept first if they fit).'''
    mandatory_total = sum(blocks[k][0] for k in mandatory_keys)
    if mandatory_total > available_budget:
        raise ValueError("Mandatory content alone exceeds budget -- escalate, do not truncate it")

    kept = list(mandatory_keys)
    remaining = available_budget - mandatory_total
    dropped = []
    for key in priority_order:
        tok = blocks[key][0]
        if tok <= remaining:
            kept.append(key)
            remaining -= tok
        else:
            dropped.append(key)
    return kept, dropped, available_budget - remaining


blocks = {
    "system": (40, request_parts["system"]),
    "ticket": (20, request_parts["ticket"]),
    "output_instructions": (15, request_parts["output_instructions"]),
    "history_turn_1": (15, request_parts["history_turn_1"]),
    "doc_sla_table": (30, request_parts["doc_sla_table"]),
    "doc_billing_faq": (50, request_parts["doc_billing_faq"]),
}
mandatory = ["system", "ticket", "output_instructions"]
BUDGET = 130

good_priority = ["history_turn_1", "doc_sla_table", "doc_billing_faq"]  # relevance-ranked
bad_priority = ["doc_billing_faq", "doc_sla_table", "history_turn_1"]   # naive, size-first

kept_good, dropped_good, used_good = select_within_budget(blocks, BUDGET, mandatory, good_priority)
kept_bad, dropped_bad, used_bad = select_within_budget(blocks, BUDGET, mandatory, bad_priority)

print("Relevance-ranked priority -> kept:", kept_good, "\n  dropped:", dropped_good)
print("\nNaive (size-first) priority -> kept:", kept_bad, "\n  dropped:", dropped_bad)


**Expected output:** the relevance-ranked run keeps both `history_turn_1` (the tier fact)
and `doc_sla_table` (the actual SLA numbers), dropping only the irrelevant billing FAQ.
The naive, size-first run spends its budget on the long but irrelevant billing FAQ first
and has to drop **both** `history_turn_1` and `doc_sla_table` — the two pieces the
question actually depends on. **Same budget, same six pieces — ranking alone decides
which request can answer the question.**

## 🛠️ The controlled lost-context demonstration

A deterministic stand-in for "the model" — not a real LLM call, so the result is
reliable and repeatable, the same mock-first principle S01's `MockModel` uses — that can
only answer using whatever text actually survived the budget cut above.


In [ ]:
SLA_BY_TIER = {"enterprise": "1 hour", "pro": "4 hours", "standard": "24 hours"}

def mock_answer_sla(kept_keys, blocks):
    # Word-boundary match -- a plain substring check would wrongly match "pro"
    # inside a word like "provided". Always match whole words in text you didn't write.
    combined_text = " ".join(blocks[k][1] for k in kept_keys).lower()
    for tier, sla in SLA_BY_TIER.items():
        if re.search(rf"\b{re.escape(tier)}\b", combined_text):
            return f"Your SLA is {sla} ({tier} tier)."
    return "I don't have enough information to determine your SLA tier."

print("Answer when the relevance-ranked context is used:")
print(" ", mock_answer_sla(kept_good, blocks))
print("\nAnswer when the naive, size-first context is used:")
print(" ", mock_answer_sla(kept_bad, blocks))


**Observation.** The relevance-ranked run answers correctly, because the fact it needs
(Enterprise tier, stated two turns ago) survived the budget cut. The naive run doesn't
fabricate a wrong tier — it honestly reports it doesn't have enough information, because
word-boundary matching correctly finds nothing, having lost both the tier mention and the
SLA table itself. **This is context loss you watched happen, with a controlled,
repeatable cause (a bad ranking decision), not a hypothetical.**

> [!IMPORTANT]
> **⚙️ Engineering implication:** "long context window" and "persistent memory" are not
> the same claim (S03 §6) — and now you've seen the concrete failure mode that confusion
> produces: a budget cut silently drops the one fact an answer depends on, unless the
> pruning policy is relevance-aware, not just size- or recency-aware by accident.

## Simple relevance ranking — without building a RAG pipeline

The `good_priority` order above wasn't arbitrary — it can be derived from a simple,
explicit keyword-overlap score between the ticket and each candidate document, no
embeddings or vector index required.


In [ ]:
def keyword_overlap_score(query_text: str, doc_text: str) -> float:
    def words(t):
        return {w.strip(".,!?:;\"'").lower() for w in t.split()}
    q, d = words(query_text), words(doc_text)
    if not d:
        return 0.0
    return len(q & d) / len(d)

ticket_text = request_parts["ticket"]
for doc_key in ["doc_sla_table", "doc_billing_faq"]:
    score = keyword_overlap_score(ticket_text, request_parts[doc_key])
    print(f"{doc_key:16s} relevance score vs. the ticket: {score:.3f}")


**Observation.** The SLA table scores higher against the ticket than the billing FAQ
does — a one-line, explainable heuristic that would have *automatically* produced the
good ranking above, without needing to hand-pick it. This is deliberately as far as this
notebook goes toward "retrieval" — a real production system facing many more candidate
documents would need embeddings and a vector index to do this at scale; that's the
subject of a future Context Engineering and RAG session, not a gap in this one.

> [!NOTE]
> **(optional)** Try the real small model instead of `mock_answer_sla`, loading it the
> same way Lab 4 will. Expect noisier, less reliable results than the deterministic mock
> above — that unreliability is itself informative, not a bug in the comparison; a 135M
> model's answer quality was never the point of the controlled demo.

### ✍️ Learner exercise

Change `BUDGET` to `160` and re-run the `select_within_budget` cell with both priority
orders. Does the outcome still differ between them? What does that tell you about how
much budget "ranking mistakes matter at all" depends on?

<details>
<summary>Instructor solution</summary>

At `BUDGET=160`, remaining budget after the mandatory 75 tokens is 85 — enough for the
naive ranking to fit `doc_billing_faq` (50) and still have room for `doc_sla_table` (30),
leaving only `history_turn_1` (15) tight but likely still fitting depending on exact
remaining room. The lesson: ranking mistakes matter most when the budget is **tight
relative to total candidate content** — a generous budget forgives a bad ranking; a tight
one (the realistic case once real documents and real history are involved) does not.

</details>


---
# Lab 4 — Streaming Architecture Decision

**Goal:** this is the first point in this notebook that needs the actual model, not just
its tokenizer — everything in Labs 1–3 ran on token counting and pure Python alone.


In [ ]:
import torch
import threading
from transformers import AutoModelForCausalLM, TextIteratorStreamer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
model.eval()
print(f"Model loaded on {device}. (This is the only model load in this entire notebook.)")


## Workload A — interactive support copilot

A human is watching this response appear, turn by turn. Streaming's actual benefit —
perceived responsiveness — applies here.


In [ ]:
def generate_with_timing(prompt_text, stream=False, max_new_tokens=60):
    chat = [{"role": "user", "content": prompt_text}]
    inputs = tokenizer.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt").to(device)
    started = time.perf_counter()

    if not stream:
        with torch.inference_mode():
            output_ids = model.generate(inputs, max_new_tokens=max_new_tokens,
                                         pad_token_id=tokenizer.eos_token_id)
        elapsed = time.perf_counter() - started
        text = tokenizer.decode(output_ids[0][inputs.shape[-1]:], skip_special_tokens=True)
        return {"text": text, "ttft_s": None, "total_s": elapsed}

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    def run():
        with torch.inference_mode():
            model.generate(inputs, max_new_tokens=max_new_tokens,
                            pad_token_id=tokenizer.eos_token_id, streamer=streamer)
    thread = threading.Thread(target=run)
    thread.start()
    first_chunk_s, pieces = None, []
    for chunk in streamer:
        if first_chunk_s is None:
            first_chunk_s = time.perf_counter() - started
        pieces.append(chunk)
    thread.join()
    total_s = time.perf_counter() - started
    return {"text": "".join(pieces), "ttft_s": first_chunk_s, "total_s": total_s}


copilot_result = generate_with_timing(
    "A customer says their invoice shows a duplicate charge. Draft a short, empathetic first reply.",
    stream=True,
)
print("Response:", copilot_result["text"])
print(f"\nTime to first visible chunk: {copilot_result['ttft_s']:.2f}s")
print(f"Total generation time:        {copilot_result['total_s']:.2f}s")


## Workload B — background ticket classification batch

Nobody is watching any single call's output appear. What matters is **total batch
throughput**, not any individual call's time-to-first-token.


In [ ]:
batch_tickets = [
    "Invoice shows a duplicate charge for last month.",
    "Cannot log in after a password reset.",
    "Where is the API documentation?",
    "Critical outage: all users seeing 503 errors.",
    "Please update the company name on our receipt.",
]

batch_started = time.perf_counter()
batch_results = []
for ticket_text in batch_tickets:
    result = generate_with_timing(
        f"Classify this support ticket in one word (billing/technical/access/general): {ticket_text}",
        stream=False, max_new_tokens=10,
    )
    batch_results.append(result)
batch_total = time.perf_counter() - batch_started

for ticket_text, result in zip(batch_tickets, batch_results):
    print(f"{ticket_text[:45]:45s} -> {result['text'].strip()!r} ({result['total_s']:.2f}s)")
print(f"\nTotal batch wall-clock time for {len(batch_tickets)} tickets: {batch_total:.2f}s")
print("No single call's time-to-first-chunk was measured here -- for a batch, it isn't the relevant number.")


**Observation.** Workload A's useful number is *time to first visible chunk* — what a
waiting human actually experiences. Workload B's useful number is *total batch time* —
what determines whether 50,000 tickets/day finish on schedule. Streaming Workload B
wouldn't be wrong, exactly — it would just measure and optimize for a number nobody is
looking at.

| | Workload A (interactive copilot) | Workload B (batch classification) |
|---|---|---|
| Useful latency metric | Time to first visible chunk | Total batch wall-clock time |
| Streaming helps? | Yes — perceived responsiveness | No — nothing is watched live |
| Operational complexity added by streaming | Chunk buffering, partial-render UI logic, reconnect handling | N/A if not streamed |
| Error handling needed | Handle an interrupted stream mid-response | Handle a failed/timed-out call, retried as a whole unit |

## Error handling for interrupted streams — a concrete failure, not a hypothetical

If a workload needs a **structured, parseable** result (S02's `TicketDecision` contract),
a stream that cuts off mid-response is not a smaller version of the answer — it's often
not parseable at all.


In [ ]:
full_json = '{"category": "billing", "priority": "high", "needs_human": true}'
truncated_json = full_json[:30]
print("What a client has received so far, mid-stream:", truncated_json)

try:
    json.loads(truncated_json)
except json.JSONDecodeError as exc:
    print(f"\nParsing the partial stream fails, as expected: {exc}")

print("\nOnce the stream completes:", json.loads(full_json))


**Observation.** The partial JSON is not "70% of a valid answer" — it's not valid at all.
A structured-output workload that streams has two real choices: buffer the entire
response before parsing it (losing most of streaming's responsiveness benefit for that
specific call), or use a provider's purpose-built structured-streaming support designed
for exactly this problem — never assume you can parse a structured result as it arrives
with ordinary JSON parsing.

> [!IMPORTANT]
> **Restating S03's core claim, now with a concrete cost attached:** streaming changes
> **delivery**, not the underlying autoregressive generation process — and for a
> structured-output workload specifically, that delivery change comes with a real
> parsing-complexity cost that a non-streaming call simply doesn't have.

### ✍️ Learner exercise

For Northstar's actual 50,000-ticket/day classification workload, would you turn
streaming on? Justify your answer using the table above, not intuition.

<details>
<summary>Instructor solution</summary>

No. Nobody watches an individual classification call's output appear — the useful metric
is total throughput across 50,000 calls/day, which streaming doesn't improve. Worse,
classification needs a structured, parseable result, and streaming adds exactly the
interrupted-parsing risk demonstrated above for no offsetting benefit. Non-streaming,
buffered, parse-once-complete is the correct choice for this specific workload.

</details>


---
# 🛠️ Conclusion exercise — Northstar's 50,000-ticket/day recommendation

Northstar wants to classify 50,000 tickets daily with **predictable latency and
spending**. Using everything from Labs 1–4, write your recommendation covering:

- **Prompt/context budget** — how much gets attached per ticket, and what gets pruned
  first if it doesn't fit.
- **Model-selection criteria** — what you'd actually test for, not just which model name.
- **Output-token limit** — and why that specific number.
- **Streaming or non-streaming** — and why.
- **Cost estimate and uncertainty** — a number from Lab 2's model, stated as an estimate
  under stated assumptions, not a measured bill.
- **Key operational risks** — the ones this notebook actually surfaced, not generic ones.

Write your recommendation in plain business language — the version you'd actually send to
Northstar's VP of Support, not an internal engineering memo.

_Write your recommendation here._


<details>
<summary>Instructor model answer</summary>

"We recommend capping each classification request to the current ticket plus the two
most relevant knowledge-base snippets and the single most recent conversation turn,
selected by keyword relevance rather than simple recency — roughly 300–400 input tokens
per ticket, well below what a brute-force 'attach everything' approach would use. Model
selection should be based on measured accuracy and critical-miss rate on a representative,
reviewed sample of your actual tickets (per S02's comparison method), not a public
benchmark ranking alone — we'd shortlist two or three candidates and test on your data
before committing. We recommend capping output to roughly 80–100 tokens, enough for a
category, priority, and a one-line justification, which controls both cost and response-
time variance. This workload should run **non-streaming**: nobody watches an individual
classification call, and a structured result that's interrupted mid-stream often can't
even be parsed — streaming would add complexity with no offsetting benefit here. Under
our stated assumptions (roughly 350 input tokens, 90 output tokens, a 5% retry rate, and
illustrative pricing of $1/$4 per million input/output tokens), the model in Lab 2
estimates a cost in the neighborhood of a few hundred dollars a month at this volume — an
estimate that depends entirely on those assumptions holding and must be re-checked against
real measured usage before being treated as a budget commitment. The main operational
risks are: a pruning policy that silently drops a fact a ticket's correct classification
actually depends on (Lab 3's demonstrated failure mode), a retry-rate spike quietly
multiplying cost (Lab 2's most expensive lever), and any assumption about provider pricing
or caching eligibility turning out to be wrong without us noticing until the invoice
arrives."

**Grading guide, if used as an assessment:** a strong answer names a *specific* budget
and pruning rule (not "use less context"), ties the model-selection criterion to measured
task performance rather than a benchmark name, gives a *reasoned* output-token number,
correctly argues non-streaming for this specific workload (not a blanket rule), states a
cost figure explicitly as assumption-dependent, and names risks that come from this
notebook's actual demonstrations rather than generic AI-safety boilerplate.

</details>


---
## ✅ Recap

- **Lab 1:** word count is not a token budget—punctuation, rare vocabulary, code, and non-Latin scripts can require more tokens.
- **Lab 2:** cost scales with input tokens, output tokens, and retries; retries multiply the entire per-attempt cost.
- **Lab 3:** a context budget forces a selection decision, and poor ranking can silently remove the fact an answer needs.
- **Lab 4:** streaming improves perceived responsiveness for an interactive user but adds delivery and error-handling complexity.

## 📚 Further reading

- `sessions/S03-llm-fundamentals/` — tokenization, context-window, and streaming mechanics.
- `sessions/S02-structured-outputs-and-model-selection/` — model comparison and typed output contracts.
- [Hugging Face tokenizer documentation](https://huggingface.co/docs/transformers/main_classes/tokenizer)
- [ipywidgets documentation](https://ipywidgets.readthedocs.io/)
